# Immortalite Zero — Self-Play Training (Colab)

Trains on a free GPU. Checkpoints save to Google Drive.

**Fresh run (v3):** cell 3 uses a new Drive folder — no resume, no rewind. Prior runs stay in `immortalite_zero_checkpoints` / `_v2`.

**Flat settings (no sims/games ramp):** 100 MCTS sims, 64 games/iter, 400 train steps, draw penalty 0.03. Self-play and gates use the same sim count and Syzygy.

**Gates (in training loop):** every 20 iters, current net vs snapshot 20 iters ago (64 games, 100 sims). First gate at iter 20 vs 0. Pass heuristic: winrate **>55%**, draws **<45/64**.

Run cells top to bottom. Set runtime to **GPU** first.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite-zero'):
    !git clone https://github.com/carlo-wong/immortalite-zero.git
%cd /content/immortalite-zero
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy tqdm

In [ ]:
# 3. Mount Google Drive for checkpoints.
from google.colab import drive
drive.mount('/content/drive')

# Fresh run: new folder so metrics/shards/checkpoints start at iter 0.
# Change RUN_TAG only when you want another clean slate (e.g. v4).
RUN_TAG = 'v3'
CKPT_DIR = f'/content/drive/MyDrive/immortalite_zero_checkpoints_{RUN_TAG}'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)
if os.path.exists(os.path.join(CKPT_DIR, 'latest.pt')):
    print('WARNING: latest.pt already in this folder — bump RUN_TAG for a true fresh start, or set resume=True in cell 6.')
else:
    print('fresh start: empty checkpoint dir (iter 0)')

In [ ]:
# 4. Confirm GPU is available and choose the train preset once.
import torch

HAS_CUDA = torch.cuda.is_available()
CUDA_DEVICE = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'
IS_T4 = HAS_CUDA and ('T4' in CUDA_DEVICE)
TRAIN_PRESET = '--device cuda --gpu' if HAS_CUDA else '--device cpu --light'

print('CUDA available:', HAS_CUDA)
print('device:', CUDA_DEVICE)
print('is_t4:', IS_T4)
print('train preset:', TRAIN_PRESET)

In [ ]:
# 5. Download Syzygy 3-4-5 WDL tablebases to local Colab disk (no Drive).
import os
import re
import urllib.parse
import urllib.request

TB_DIR = '/content/syzygy345'
TB_MIRRORS = [
    'http://tablebase.sesse.net/syzygy/3-4-5/',
]
os.makedirs(TB_DIR, exist_ok=True)

index_html = None
TB_BASE_URL = None
for mirror in TB_MIRRORS:
    try:
        index_html = urllib.request.urlopen(mirror, timeout=30).read().decode('utf-8', errors='ignore')
        TB_BASE_URL = mirror
        break
    except Exception as exc:
        print(f'Failed mirror {mirror}: {exc}')

if index_html is None or TB_BASE_URL is None:
    raise RuntimeError('Could not reach any Syzygy mirror for 3-4-5 WDL files.')

files = sorted(set(re.findall(r'href="([^"/]+\.rtbw)"', index_html)))
if not files:
    raise RuntimeError(f'No .rtbw files found at Syzygy source URL: {TB_BASE_URL}')

missing = [name for name in files if not os.path.exists(os.path.join(TB_DIR, name))]
print(f'Syzygy WDL source -> {TB_BASE_URL}')
print(f'Syzygy WDL files: {len(files)} total, {len(missing)} missing in this session')

for idx, name in enumerate(missing, start=1):
    src = urllib.parse.urljoin(TB_BASE_URL, name)
    dst = os.path.join(TB_DIR, name)
    urllib.request.urlretrieve(src, dst)
    if idx % 25 == 0 or idx == len(missing):
        print(f'Downloaded {idx}/{len(missing)} missing files')

print('Syzygy directory ->', TB_DIR)

In [ ]:
# 6. Config + train (flat settings, in-loop gates). Edit TRAIN below only.
import os

if 'TRAIN_PRESET' not in globals():
    raise RuntimeError('Run cell 4 first.')

TRAIN = {
    'sims': 100,
    'gate_sims': 100,
    'games': 64,
    'train_steps': 400,
    'concurrency': 64,
    'draw_penalty': 0.03,
    'gate_every': 20,
    'gate_games': 64,
    'save_every': 10,
    'iterations': 100,
    'resume': False,   # fresh run: keep False (no --resume, no rewind)
    'resign': False,
}
RESIGN_THRESHOLD = -0.90
RESIGN_PLIES = 3
RESIGN_MIN_MOVES = 20

resume_arg = ''
if TRAIN['resume']:
    resume_path = os.path.join(CKPT_DIR, 'latest.pt')
    if os.path.exists(resume_path):
        resume_arg = f'--resume {resume_path}'
    else:
        print('WARNING: resume=True but no latest.pt — starting at iter 0')

syzygy_arg = f'--syzygy-path "{TB_DIR}"'
resign_arg = ''
if TRAIN['resign']:
    resign_arg = (
        f'--resign-threshold {RESIGN_THRESHOLD} '
        f'--resign-plies {RESIGN_PLIES} '
        f'--resign-min-moves {RESIGN_MIN_MOVES}'
    )

print('TRAIN config:', TRAIN)
print(f'preset={TRAIN_PRESET} ckpt_dir={CKPT_DIR}')

!python -m engine.train --iterations {TRAIN['iterations']} {TRAIN_PRESET} --games {TRAIN['games']} --train-steps {TRAIN['train_steps']} --concurrency {TRAIN['concurrency']} --sims {TRAIN['sims']} --draw-penalty {TRAIN['draw_penalty']} {resign_arg} {syzygy_arg} --save-every {TRAIN['save_every']} --gate-every {TRAIN['gate_every']} --gate-games {TRAIN['gate_games']} --gate-sims {TRAIN['gate_sims']} --gate-exploration-moves 20 --checkpoint-dir "$CKPT_DIR" {resume_arg}

In [ ]:
# 7. Manual gate (optional) — same settings as in-loop gates in cell 6.
import os
import chess.syzygy
import torch
from engine.config import Config, NetConfig
from engine.network import ChessNet
from engine.train import play_match, _log_gate_metrics

if 'TRAIN' not in globals():
    raise RuntimeError('Run cell 6 first (defines TRAIN).')

# Edit after training progresses (e.g. 20 vs 0 after first gate).
CHECKPOINT_A = 20
CHECKPOINT_B = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
cfg.train.selfplay_concurrency = TRAIN['concurrency']
cfg.train.draw_penalty = TRAIN['draw_penalty']
cfg.train.syzygy_path = TB_DIR
tablebase = chess.syzygy.open_tablebase(TB_DIR) if os.path.isdir(TB_DIR) else None


def _checkpoint_iteration(checkpoint_ref, state):
    if checkpoint_ref == 'latest':
        if isinstance(state, dict):
            return int(state.get('iteration', -1))
        return -1
    try:
        return int(checkpoint_ref)
    except (TypeError, ValueError):
        return -1


# Resolve Path A
if CHECKPOINT_A == 'latest':
    path_a = os.path.join(CKPT_DIR, 'latest.pt')
    label_a = "Latest"
else:
    path_a = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_A:04d}.pt')
    label_a = f"Iter {CHECKPOINT_A}"

# Resolve Path B
if CHECKPOINT_B == 'latest':
    path_b = os.path.join(CKPT_DIR, 'latest.pt')
    label_b = "Latest"
else:
    path_b = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_B:04d}.pt')
    label_b = f"Iter {CHECKPOINT_B}"

if not os.path.exists(path_a) or not os.path.exists(path_b):
    print(f"Error: Make sure both checkpoints exist!\nPath A: {path_a}\nPath B: {path_b}")
else:
    print(f"Loading Checkpoint A ({label_a}) from {path_a}...")
    state_a = torch.load(path_a, map_location=DEVICE)
    from engine.config import NetConfig
    net_cfg_a = cfg.net
    if isinstance(state_a, dict) and 'net' in state_a:
        net_cfg_a = NetConfig(**state_a['net'])
    net_a = ChessNet(net_cfg_a).to(DEVICE)
    net_a.load_state_dict(state_a['model'] if isinstance(state_a, dict) and 'model' in state_a else state_a, strict=False)
    net_a.eval()

    print(f"Loading Checkpoint B ({label_b}) from {path_b}...")
    state_b = torch.load(path_b, map_location=DEVICE)
    net_cfg_b = cfg.net
    if isinstance(state_b, dict) and 'net' in state_b:
        net_cfg_b = NetConfig(**state_b['net'])
    net_b = ChessNet(net_cfg_b).to(DEVICE)
    net_b.load_state_dict(state_b['model'] if isinstance(state_b, dict) and 'model' in state_b else state_b, strict=False)
    net_b.eval()

    print(f"\nStarting Match: {label_a} vs {label_b} ({TRAIN['gate_games']} games, {TRAIN['gate_sims']} sims)...")

    metrics = play_match(
        net_a, net_b, cfg,
        n_games=TRAIN['gate_games'],
        sims=TRAIN['gate_sims'],
        device=DEVICE,
        tablebase=tablebase,
    )
    if tablebase is not None:
        tablebase.close()
    winrate = metrics['winrate']
    wins = metrics['wins_as_white'] + metrics['wins_as_black']
    losses = metrics['losses_as_white'] + metrics['losses_as_black']
    draws = metrics['draws_as_white'] + metrics['draws_as_black']

    iter_a = _checkpoint_iteration(CHECKPOINT_A, state_a)
    iter_b = _checkpoint_iteration(CHECKPOINT_B, state_b)
    _log_gate_metrics(CKPT_DIR, iter_a, iter_b, metrics, TRAIN['gate_games'])

    wdl_notation = f"+{wins} ={draws} -{losses}"

    print("\n" + "="*40)
    print(f"MATCH COMPLETED!")
    print(f"{label_a} score vs {label_b}: {winrate:.3f} [{wdl_notation}]")
    print(f"  As White: Wins: {metrics['wins_as_white']}, Losses: {metrics['losses_as_white']}, Draws: {metrics['draws_as_white']}")
    print(f"  As Black: Wins: {metrics['wins_as_black']}, Losses: {metrics['losses_as_black']}, Draws: {metrics['draws_as_black']}")
    print(f"  Mean Game Length: {metrics['mean_game_len']:.1f} plies")
    print(f"  Terminations: {metrics['terminations']}")
    print(f"  Logged to: {os.path.join(CKPT_DIR, 'metrics_gates.csv')}")
    if winrate > 0.55:
        print(f"Result: {label_a} PASSED the gate (Stronger than {label_b})!")
    elif winrate < 0.45:
        print(f"Result: {label_a} is significantly WEAKER than {label_b}!")
    else:
        print(f"Result: {label_a} and {label_b} are roughly EQUAL (within noise margin).")
    print("="*40)

In [ ]:
# 8. Plot training progress (run AFTER at least one training iteration finishes).
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 6 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if {'train_steps', 'batch_size', 'samples'}.issubset(df.columns):
                df['reuse_ratio'] = (
                    df['train_steps'] * df['batch_size'] /
                    df['samples'].replace(0, float('nan'))
                )

            # Gate results live in a separate file (logged independently of
            # metrics.csv so a gate crash never loses per-iteration metrics).
            gates_path = os.path.join(CKPT_DIR, 'metrics_gates.csv')
            gates_df = pd.read_csv(gates_path) if os.path.exists(gates_path) else None
            if gates_df is not None and not gates_df.empty:
                gates_df['iter'] = pd.to_numeric(gates_df['iter'], errors='coerce')
                gates_df['winrate'] = pd.to_numeric(gates_df['winrate'], errors='coerce')
                gates_df = gates_df.dropna(subset=['iter']).copy()

            # Early-run health checks (first 10-20 iterations of a fresh run).
            df_window = df[df['iter'] <= 20].copy()
            if len(df_window) >= 10:
                entropy_ok = float(df_window['policy_entropy'].iloc[-1]) >= 1.8
                sign_acc_series = df_window['value_sign_acc'].dropna()
                value_ok = (not sign_acc_series.empty) and float(sign_acc_series.iloc[-1]) >= 0.6
                gate_rows = int((gates_df['iter'] <= 20).sum()) if gates_df is not None and not gates_df.empty else 0
                gate_ok = gate_rows >= 1
                reuse_series = df_window['reuse_ratio'].dropna() if 'reuse_ratio' in df_window.columns else pd.Series(dtype=float)
                reuse_ok = (not reuse_series.empty) and float(reuse_series.iloc[-1]) <= 20.0
                print('Early-run checks (iter <= 20):')
                print(f"  policy_entropy >= 1.8: {entropy_ok}")
                print(f"  value_sign_acc >= 0.6: {value_ok}")
                print(f"  gate rows present: {gate_ok} (rows={gate_rows})")
                print(f"  reuse_ratio <= 20x: {reuse_ok}")
            else:
                print('Early-run checks need at least 10 completed iterations.')

            fig, ax = plt.subplots(3, 3, figsize=(18, 12))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if gates_df is None or gates_df.empty:
                ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                ax[5].set_title('Strength gate')
                ax[5].set_xlabel('iteration')
            else:
                gates_df.plot(
                    x='iter',
                    y='winrate',
                    marker='o',
                    ax=ax[5],
                    title='Strength gate (vs -20 iters)',
                    legend=False,
                )
                ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)

            plot_panel(ax[6], ['learning_rate'], 'Learning rate')
            plot_panel(ax[7], ['reuse_ratio'], 'Reuse ratio (train load / fresh samples)')
            plot_panel(ax[8], ['buffer_size'], 'Replay buffer size')

            plt.tight_layout()
            plt.show()

In [ ]:
# Separator cell (safe no-op).

## After training
Download `latest.pt` from your Drive folder and run the analysis server locally:

```bash
set IMMORTALITE_ZERO_CHECKPOINT=path\to\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```
Then open http://localhost:8000/app/ .